In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.signal import stft
from scipy.ndimage import uniform_filter1d, binary_erosion, binary_dilation, label
from scipy.stats import norm
from skimage.filters import threshold_otsu
from scipy.ndimage import gaussian_filter1d


import os

from funciones.cargador import obtener_una_muestra, DATA_DIR

# Método nperseg único

In [ ]:
def detector_entropia_ciego(iq_tensor: torch.Tensor, fs: float = 14e6, nperseg: int = 1024):
    """
    Detector ciego basado en Entropía Espectral de Shannon.
    No requiere conocer la modulación, ancho de banda ni frecuencia central de la señal.
    """
    # 1. Extracción y conversión a complejo
    i_np = iq_tensor[0].numpy()
    q_np = iq_tensor[1].numpy()
    signal_complex = i_np + 1j * q_np
    
    # 2. Espectrograma de alta resolución temporal
    # nperseg define el trade-off: 1024 a 14MHz da ~73 microsegundos por salto
    f, t, Zxx = stft(signal_complex, fs=fs, nperseg=nperseg, return_onesided=False)
    
    # 3. Densidad de Potencia (Magnitud al cuadrado)
    Pxx = np.abs(Zxx)**2
    
    # 4. Normalización a Distribución de Probabilidad (por cada instante de tiempo)
    suma_potencia = np.sum(Pxx, axis=0)
    # Evitamos divisiones por cero en zonas de silencios absolutos del SDR
    suma_potencia[suma_potencia == 0] = 1e-12 
    prob_dist = Pxx / suma_potencia
    
    # Evitamos log(0) sumando un epsilon
    prob_dist = prob_dist + 1e-12
    
    # 5. Cálculo de la Entropía de Shannon H(t) = - sum(p * log2(p))
    entropia = -np.sum(prob_dist * np.log2(prob_dist), axis=0)
    
    return t * 1000, entropia  # Devolvemos tiempo en ms y la entropía

In [ ]:
# 1. CARGA DE LA MUESTRA
# Usamos la misma muestra que te generó la gráfica anterior
iq_dron, _, _, _ = obtener_una_muestra(DATA_DIR, target=2, snr=-4, index=5)

# 2. PROCESAMIENTO MULTI-DOMINIO
fs = 14e6
i_np = iq_dron[0].numpy()
q_np = iq_dron[1].numpy()
signal_complex = i_np + 1j * q_np

# A) Dominio Temporal: Envolvente de Amplitud
amplitud = np.abs(signal_complex)
tiempo_amp_ms = np.arange(len(amplitud)) / fs * 1000

# B) Dominio Tiempo-Frecuencia: STFT (Espectrograma)
nperseg = 1024
f, t, Zxx = stft(signal_complex, fs=fs, nperseg=nperseg, return_onesided=False)

# Reordenamos las frecuencias para centrar el DC en el espectrograma (de -fs/2 a fs/2)
f_shifted = np.fft.fftshift(f)
Zxx_shifted = np.fft.fftshift(Zxx, axes=0)

Pxx_db = 10 * np.log10(np.abs(Zxx_shifted)**2 + 1e-12)
tiempo_stft_ms = t * 1000

# C) Dominio de Incertidumbre: Entropía Espectral de Shannon
Pxx_orig = np.abs(Zxx)**2
suma_potencia = np.sum(Pxx_orig, axis=0)
suma_potencia[suma_potencia == 0] = 1e-12 

prob_dist = Pxx_orig / suma_potencia
prob_dist = prob_dist + 1e-12
entropia = -np.sum(prob_dist * np.log2(prob_dist), axis=0)

# 3. VISUALIZACIÓN SIGINT 3-PANEL
plt.style.use('dark_background')
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12), sharex=True, 
                                    gridspec_kw={'height_ratios': [1, 2, 1.5]})
fig.patch.set_facecolor('#1a1a2e')

for ax in [ax1, ax2, ax3]:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='#e0e0e0')
    for spine in ax.spines.values(): spine.set_color('#e0e0e0')

# Panel 1: Amplitud (Submuestreamos x10 para acelerar el renderizado sin perder visualización)
step = 10
ax1.plot(tiempo_amp_ms[::step], amplitud[::step], color='#9b59b6', linewidth=0.5)
ax1.set_title('1. Dominio Temporal: Amplitud de la Señal (|I+jQ|)', color='white', fontsize=13)
ax1.set_ylabel('Magnitud', color='white')
ax1.grid(color='#2c3e6b', alpha=0.4)

# Panel 2: Espectrograma
mesh = ax2.pcolormesh(tiempo_stft_ms, f_shifted / 1e6, Pxx_db, shading='gouraud', cmap='viridis')
ax2.set_title('2. Dominio Tiempo-Frecuencia: Espectrograma (STFT)', color='white', fontsize=13)
ax2.set_ylabel('Frecuencia (MHz)', color='white')

# Panel 3: Entropía Espectral
ax3.plot(tiempo_stft_ms, entropia, color='#2ecc71', linewidth=1.5, label='Entropía Espectral')
umbral = np.median(entropia) - np.std(entropia)
ax3.axhline(y=umbral, color='#e74c3c', linestyle='--', linewidth=2, label='Umbral de Trigger')
ax3.set_title('3. Dominio de Incertidumbre: Entropía Espectral de Shannon', color='white', fontsize=13)
ax3.set_xlabel('Tiempo (ms)', color='white', fontsize=12)
ax3.set_ylabel('Entropía (Bits)', color='white')
ax3.grid(color='#2c3e6b', alpha=0.4)
ax3.legend(facecolor='#1a1a2e', edgecolor='#e0e0e0', labelcolor='white')

plt.tight_layout()
plt.show()

# Prueba multiescala

In [ ]:
def detector_entropia_multiscala(
    iq_tensor: torch.Tensor,
    fs: float = 14e6,
    escalas: dict = None,
    sigma: float = 1.0,
    suavizado_frames: int = 5,
    min_duracion_frames: int = 3,
):
    """
    Detector de transmisión basado en Entropía Espectral Multiscala.
    Parámetros clave:
      escalas             : dict {nombre: nperseg} — qué resoluciones usar
      sigma               : sensibilidad del umbral (↑ = menos sensible, menos Pfa)
      suavizado_frames    : ventana de media móvil sobre H(m) antes de fusionar
                            (elimina picos de 1-2 frames por fluctuación estadística)
      min_duracion_frames : un evento solo cuenta si dura ≥ N frames consecutivos
                            (equivale a una duración mínima de burst)
    """
    if escalas is None:
        escalas = {
            "gruesa": 4096,   # 292 µs/frame, 3.4 kHz/bin
            "media":  1024,   #  73 µs/frame, 13.7 kHz/bin
        }
    signal = iq_tensor[0].numpy() + 1j * iq_tensor[1].numpy()
    curvas = {}
    for nombre, nperseg in escalas.items():
        _, t, Zxx = stft(signal, fs=fs, nperseg=nperseg, return_onesided=False)
        Pxx  = np.abs(Zxx) ** 2
        suma = Pxx.sum(axis=0)
        suma[suma == 0] = 1e-12
        prob = Pxx / suma + 1e-12
        H    = -(prob * np.log2(prob)).sum(axis=0)
        # Suavizado temporal: media móvil sobre los frames de entropía
        if suavizado_frames > 1:
            H = uniform_filter1d(H, size=suavizado_frames)
        curvas[nombre] = (t * 1000, H)
    # Interpolar a la rejilla de la escala media (o la más densa disponible)
    t_ref_nombre = min(escalas, key=lambda n: escalas[n])
    t_ref = curvas[t_ref_nombre][0]
    H_interp = {
        nombre: np.interp(t_ref, t_i, H_i)
        for nombre, (t_i, H_i) in curvas.items()
    }
    H_min = np.minimum.reduce(list(H_interp.values()))
    # Umbral adaptativo
    umbral = np.median(H_min) - sigma * np.std(H_min)
    # Filtro de duración mínima: rechaza dips de < min_duracion_frames consecutivos
    por_debajo = H_min < umbral
    if min_duracion_frames > 1:
        from scipy.ndimage import binary_erosion
        estructura = np.ones(min_duracion_frames, dtype=bool)
        por_debajo = binary_erosion(por_debajo, structure=estructura)
    hay_transmision = bool(np.any(por_debajo))
    return t_ref, H_min, H_interp, umbral, por_debajo, hay_transmision

In [ ]:
# ─── PARÁMETROS ──────────────────────────────────────────────────────────────
TARGET         = 0
SNR            = -10
INDEX          = 0
SIGMA          = 1.5       # desviaciones típicas (relativo)
MIN_DROP_BITS  = 0.5       # caída mínima absoluta en bits — CLAVE para bajo SNR
SUAVIZADO      = 7         # frames de media móvil
MIN_DUR        = 4         # frames mínimos para burst válido
MAX_GAP        = 10        # frames — huecos menores se fusionan
MARGEN         = 3         # frames de margen al recortar
ESCALAS        = {"gruesa": 4096, "media": 1024}
FS             = 14e6
# ─────────────────────────────────────────────────────────────────────────────
iq, _, _, _ = obtener_una_muestra(DATA_DIR, target=TARGET, snr=SNR, index=INDEX)
signal = iq[0].numpy() + 1j * iq[1].numpy()
# ── 1. ENTROPÍA MULTISCALA ────────────────────────────────────────────────────
def _entropia(signal, fs, nperseg, suavizado):
    _, t, Zxx = stft(signal, fs=fs, nperseg=nperseg, return_onesided=False)
    Pxx = np.abs(Zxx) ** 2
    suma = Pxx.sum(axis=0); suma[suma == 0] = 1e-12
    prob = Pxx / suma + 1e-12
    H = -(prob * np.log2(prob)).sum(axis=0)
    return t * 1000, uniform_filter1d(H, size=suavizado) if suavizado > 1 else H
curvas = {n: _entropia(signal, FS, np, SUAVIZADO) for n, np in ESCALAS.items()}
t_ref  = curvas["media"][0]
H_int  = {n: np.interp(t_ref, ti, Hi) for n, (ti, Hi) in curvas.items()}
H_min  = np.minimum.reduce(list(H_int.values()))
# ── 2. UMBRAL ADAPTATIVO DOBLE ────────────────────────────────────────────────
med = np.median(H_min)
std = np.std(H_min)
umbral_rel = med - SIGMA * std          # relativo a la varianza del archivo
umbral_abs = med - MIN_DROP_BITS        # piso absoluto de caída mínima
umbral = min(umbral_rel, umbral_abs)    # el más restrictivo
print(f"Mediana H: {med:.4f} bits")
print(f"Std     H: {std:.4f} bits")
print(f"Umbral relativo (σ): {umbral_rel:.4f}")
print(f"Umbral absoluto    : {umbral_abs:.4f}")
print(f"Umbral final       : {umbral:.4f}  ← {'relativo' if umbral_rel < umbral_abs else 'ABSOLUTO'} domina")
# ── 3. SEGMENTACIÓN ───────────────────────────────────────────────────────────
mascara  = binary_erosion(H_min < umbral, structure=np.ones(MIN_DUR))
merged   = binary_dilation(mascara, structure=np.ones(MAX_GAP))
labeled_arr, n_bursts = label(merged)
bursts = []
for i in range(1, n_bursts + 1):
    idx = np.where(labeled_arr == i)[0]
    i0  = max(0, idx[0] - MARGEN)
    i1  = min(len(t_ref) - 1, idx[-1] + MARGEN)
    duracion = t_ref[i1] - t_ref[i0]
    profundidad = med - H_min[i0:i1+1].min()
    bursts.append((t_ref[i0], t_ref[i1], i0, i1, duracion, profundidad))
print(f"\nBursts detectados: {n_bursts}")
for k, (t0, t1, _, _, dur, prof) in enumerate(bursts):
    print(f"  B{k+1:02d}: {t0:6.2f}–{t1:6.2f} ms | dur={dur:.2f} ms | caída={prof:.2f} bits")
# ── 4. VISUALIZACIÓN ──────────────────────────────────────────────────────────
f_st, t_st, Zxx = stft(signal, fs=FS, nperseg=1024, return_onesided=False)
f_sh   = np.fft.fftshift(f_st)
Pdb    = 10 * np.log10(np.abs(np.fft.fftshift(Zxx, axes=0))**2 + 1e-12)
tsms   = t_st * 1000
amp    = np.abs(signal)
tamp   = np.arange(len(amp)) / FS * 1000
COLS = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6','#1abc9c','#e91e63']
plt.style.use('dark_background')
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True,
                         gridspec_kw={'height_ratios': [1, 2, 1.2, 1.5]})
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='#e0e0e0')
    for sp in ax.spines.values(): sp.set_color('#444')
# Amplitud
axes[0].plot(tamp[::10], amp[::10], color='#9b59b6', lw=0.5)
for k, (t0, t1, *_ ) in enumerate(bursts):
    col = COLS[k % len(COLS)]
    axes[0].axvspan(t0, t1, color=col, alpha=0.25)
    axes[0].text((t0+t1)/2, amp.max()*0.85, f'B{k+1}',
                 color=col, ha='center', fontsize=7, fontweight='bold')
axes[0].set_title('Amplitud  |I+jQ|', color='white', fontsize=12)
axes[0].set_ylabel('Magnitud', color='white')
axes[0].grid(color='#2c3e6b', alpha=0.3)
# Espectrograma
axes[1].pcolormesh(tsms, f_sh/1e6, Pdb, shading='gouraud', cmap='viridis')
for k, (t0, t1, *_) in enumerate(bursts):
    axes[1].axvspan(t0, t1, color=COLS[k % len(COLS)], alpha=0.2)
axes[1].set_title('Espectrograma (nperseg=1024)', color='white', fontsize=12)
axes[1].set_ylabel('Frecuencia (MHz)', color='white')
# Entropía por escala
ec = {'gruesa': '#e74c3c', 'media': '#f39c12'}
for n, Hi in H_int.items():
    axes[2].plot(t_ref, Hi, color=ec[n], lw=1.2, label=f'{n} ({ESCALAS[n]})')
axes[2].axhline(umbral_rel, color='white', ls=':', lw=0.8, alpha=0.5, label=f'Umbral relativo ({umbral_rel:.3f})')
axes[2].axhline(umbral_abs, color='cyan',  ls=':', lw=0.8, alpha=0.5, label=f'Umbral absoluto ({umbral_abs:.3f})')
axes[2].axhline(umbral,     color='red',   ls='--', lw=1.2,            label=f'Umbral final ({umbral:.3f})')
axes[2].set_title('Entropía por escala (suavizada)', color='white', fontsize=12)
axes[2].set_ylabel('Bits', color='white')
axes[2].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8)
axes[2].grid(color='#2c3e6b', alpha=0.3)
# Fusión + bursts
axes[3].plot(t_ref, H_min, color='#2ecc71', lw=1.8, label='H_mín fusión')
axes[3].axhline(umbral, color='#e74c3c', ls='--', lw=1.5, label=f'Umbral final')
for k, (t0, t1, i0, i1, dur, prof) in enumerate(bursts):
    col = COLS[k % len(COLS)]
    axes[3].fill_between(t_ref[i0:i1+1], H_min.min()-0.2, umbral,
                         where=H_min[i0:i1+1] < umbral,
                         color=col, alpha=0.45, label=f'B{k+1} {dur:.1f}ms ↓{prof:.1f}b')
axes[3].set_title('Fusión Multiscala — Bursts segmentados', color='white', fontsize=12)
axes[3].set_xlabel('Tiempo (ms)', color='white', fontsize=11)
axes[3].set_ylabel('Bits', color='white')
axes[3].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8,
               ncol=min(4, max(1, n_bursts//2 + 1)), loc='lower left')
axes[3].grid(color='#2c3e6b', alpha=0.3)
estado = f"✓ {n_bursts} burst{'s' if n_bursts != 1 else ''}" if n_bursts > 0 else "✗ Sin transmisión"
fig.suptitle(f'target={TARGET} | SNR={SNR} dB | {estado} | '
             f'σ={SIGMA} | minDrop={MIN_DROP_BITS}b | suav={SUAVIZADO}f | minDur={MIN_DUR}f | maxGap={MAX_GAP}f',
             color='white', fontsize=12, y=1.005)
plt.tight_layout()
plt.show()

In [ ]:
# ─── SOLO 6 PARÁMETROS, todos con significado físico directo ─────────────────
TARGET  = 1
SNR     = -4
INDEX   = 5
NPERSEG      = 4096   # Δf=3.42 kHz/bin | Δt=0.29 ms/frame
Z_THRESH     = 3.0    # z-score → Pfa teórica por frame controlada
REF_CELLS    = 80     # frames de referencia a cada lado para estimar ruido local
GUARD_CELLS  = 8      # frames de guardia (excluidos del cálculo de ruido)
MIN_BURST_MS = 1.0    # duración mínima aceptable de burst (ms)
MERGE_GAP_MS = 5.0    # fusionar bursts si están más cerca que esto (ms)
FS           = 14e6
# ─────────────────────────────────────────────────────────────────────────────
iq, _, _, _ = obtener_una_muestra(DATA_DIR, target=TARGET, snr=SNR, index=INDEX)
signal = iq[0].numpy() + 1j * iq[1].numpy()
# ── 1. ENTROPÍA ESPECTRAL ─────────────────────────────────────────────────────
_, t_h, Zxx = stft(signal, fs=FS, nperseg=NPERSEG, return_onesided=False)
Pxx  = np.abs(Zxx) ** 2
suma = Pxx.sum(axis=0); suma[suma == 0] = 1e-12
prob = Pxx / suma + 1e-12
H    = -(prob * np.log2(prob)).sum(axis=0)
t_ms = t_h * 1000
dt_ms = float(t_ms[1] - t_ms[0])
n     = len(H)
# ── 2. CA-CFAR SOBRE H(m) ─────────────────────────────────────────────────────
# Para cada frame m, estima μ y σ del ruido LOCAL a partir de las celdas
# de referencia (excluyendo la guardia). El umbral se adapta automáticamente.
pad   = REF_CELLS + GUARD_CELLS
H_pad = np.pad(H, (pad, pad), mode='reflect')
z_arr  = np.zeros(n)
mu_arr = np.zeros(n)
sg_arr = np.zeros(n)
for m in range(n):
    p         = m + pad
    left_ref  = H_pad[p - GUARD_CELLS - REF_CELLS : p - GUARD_CELLS]
    right_ref = H_pad[p + GUARD_CELLS + 1 : p + GUARD_CELLS + 1 + REF_CELLS]
    ref       = np.concatenate([left_ref, right_ref])
    mu        = ref.mean()
    sg        = ref.std()
    mu_arr[m] = mu
    sg_arr[m] = sg
    z_arr[m]  = (H[m] - mu) / (sg + 1e-10)
mascara_raw = z_arr < -Z_THRESH
# ── 3. POST-PROCESADO EN MS (no en frames) ────────────────────────────────────
min_frames   = max(1, round(MIN_BURST_MS / dt_ms))
merge_frames = max(1, round(MERGE_GAP_MS / dt_ms))
mascara_filt = binary_erosion(mascara_raw,  structure=np.ones(min_frames))
merged       = binary_dilation(mascara_filt, structure=np.ones(merge_frames))
labeled_arr, n_bursts = label(merged)
bursts = []
for i in range(1, n_bursts + 1):
    idx  = np.where(labeled_arr == i)[0]
    i0, i1 = idx[0], idx[-1]
    bursts.append({
        "t0": t_ms[i0],  "t1": t_ms[i1],
        "i0": i0,        "i1": i1,
        "dur_ms": t_ms[i1] - t_ms[i0],
        "drop_b": mu_arr[i0:i1+1].mean() - H[i0:i1+1].min(),
        "z_min":  z_arr[i0:i1+1].min(),
    })
# Diagnóstico
pfa = norm.sf(Z_THRESH)
print(f"Δt/frame  : {dt_ms:.3f} ms      Δf/bin : {FS/NPERSEG/1e3:.2f} kHz")
print(f"n frames  : {n}")
print(f"Referencia: {REF_CELLS} frames = {REF_CELLS*dt_ms:.1f} ms a cada lado")
print(f"Guardia   : {GUARD_CELLS} frames = {GUARD_CELLS*dt_ms:.1f} ms a cada lado")
print(f"z_thresh  : {Z_THRESH}  →  Pfa teórica/frame: {pfa*100:.4f}%")
print(f"Min burst : {MIN_BURST_MS} ms = {min_frames} frames")
print(f"Merge gap : {MERGE_GAP_MS} ms = {merge_frames} frames")
print(f"\nBursts: {n_bursts}")
for k, b in enumerate(bursts):
    print(f"  B{k+1:02d}: {b['t0']:6.2f}–{b['t1']:6.2f} ms  "
          f"dur={b['dur_ms']:.1f}ms  ↓{b['drop_b']:.3f}b  z_min={b['z_min']:.2f}")
# ── 4. VISUALIZACIÓN ──────────────────────────────────────────────────────────
amp  = np.abs(signal)
tamp = np.arange(len(amp)) / FS * 1000
COLS = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6','#1abc9c','#ff69b4']
def marcar(ax, alpha=0.2, yref=None):
    for k, b in enumerate(bursts):
        col = COLS[k % len(COLS)]
        ax.axvspan(b["t0"], b["t1"], color=col, alpha=alpha)
        if yref is not None:
            ax.text((b["t0"]+b["t1"])/2, yref, f'B{k+1}',
                    color=col, ha='center', fontsize=8, fontweight='bold')
plt.style.use('dark_background')
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True,
                         gridspec_kw={'height_ratios': [1, 2, 1.5, 1.5]})
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='#e0e0e0')
    for sp in ax.spines.values(): sp.set_color('#444')
# P1: Amplitud
axes[0].plot(tamp[::8], amp[::8], color='#9b59b6', lw=0.4)
marcar(axes[0], alpha=0.3, yref=amp.max() * 0.85)
axes[0].set_title('Amplitud  |I+jQ|', color='white', fontsize=12)
axes[0].set_ylabel('Magnitud', color='white')
axes[0].grid(color='#2c3e6b', alpha=0.3)
# P2: Espectrograma con la misma resolución que el detector
f_st, t_st, Zxx_sp = stft(signal, fs=FS, nperseg=NPERSEG, return_onesided=False)
f_sh = np.fft.fftshift(f_st)
Pdb  = 10 * np.log10(np.abs(np.fft.fftshift(Zxx_sp, axes=0))**2 + 1e-12)
axes[1].pcolormesh(t_st*1000, f_sh/1e6, Pdb, shading='gouraud', cmap='viridis')
marcar(axes[1], alpha=0.25)
axes[1].set_title(f'Espectrograma  (nperseg={NPERSEG} | '
                  f'Δf={FS/NPERSEG/1e3:.2f} kHz/bin | Δt={dt_ms:.2f} ms/frame)',
                  color='white', fontsize=12)
axes[1].set_ylabel('Frecuencia (MHz)', color='white')
# P3: H(m) + μ_local + zona de detección CFAR
axes[2].fill_between(t_ms, mu_arr - Z_THRESH * sg_arr, mu_arr,
                     alpha=0.12, color='cyan', label=f'Zona detección  z < −{Z_THRESH}')
axes[2].plot(t_ms, mu_arr, color='white', lw=0.8, ls='--', alpha=0.55,
             label='μ local (ventana CFAR)')
axes[2].plot(t_ms, mu_arr - Z_THRESH * sg_arr, color='cyan', lw=0.8, ls=':', alpha=0.55)
axes[2].plot(t_ms, H, color='#2ecc71', lw=1.5, label='H(m)')
axes[2].fill_between(t_ms, H, mu_arr - Z_THRESH * sg_arr,
                     where=z_arr < -Z_THRESH,
                     color='#e74c3c', alpha=0.55, label='Detección activa')
marcar(axes[2], alpha=0.12)
axes[2].set_title('Entropía H(m)  +  umbral CFAR adaptativo local', color='white', fontsize=12)
axes[2].set_ylabel('Bits', color='white')
axes[2].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8)
axes[2].grid(color='#2c3e6b', alpha=0.3)
# P4: z-score — la señal de decisión limpia
axes[3].axhline(-Z_THRESH, color='#e74c3c', ls='--', lw=1.5,
                label=f'Umbral  z = −{Z_THRESH}  (Pfa/frame = {pfa*100:.4f}%)')
axes[3].axhline(0, color='white', ls=':', lw=0.5, alpha=0.4)
axes[3].plot(t_ms, z_arr, color='#f39c12', lw=1.0, label='z-score(m)')
axes[3].fill_between(t_ms, z_arr, -Z_THRESH,
                     where=z_arr < -Z_THRESH,
                     color='#e74c3c', alpha=0.45, label='Burst detectado')
marcar(axes[3], alpha=0.12)
axes[3].set_title('z-score CFAR  (negativo intenso = señal bajo el nivel de ruido local)',
                  color='white', fontsize=12)
axes[3].set_xlabel('Tiempo (ms)', color='white', fontsize=11)
axes[3].set_ylabel('σ', color='white')
axes[3].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8)
axes[3].grid(color='#2c3e6b', alpha=0.3)
estado = f"✓ {n_bursts} burst{'s' if n_bursts != 1 else ''}" if n_bursts else "✗ Sin transmisión"
fig.suptitle(f'CA-CFAR  |  target={TARGET}  SNR={SNR} dB  |  {estado}  |  '
             f'nperseg={NPERSEG}  z={Z_THRESH}  ref={REF_CELLS}f  guard={GUARD_CELLS}f  '
             f'minBurst={MIN_BURST_MS}ms  merge={MERGE_GAP_MS}ms',
             color='white', fontsize=11, y=1.005)
plt.tight_layout()
plt.show()

# Última versión

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PARÁMETROS
# ══════════════════════════════════════════════════════════════════════════════
TARGET  = 5;  SNR = -20;  INDEX = 18
FS           = 14e6
NPERSEG      = 2048
Z_THRESH     = 3      # más sensible — el filtro de ancho espectral compensa los FA
MIN_BURST_MS = 0.15
MERGE_GAP_MS = 1.0
MIN_Z_ABS    = 5.0
# Discriminador FHSS vs. WiFi:
# P(bin_ruido > BG_MULT × fondo) = 2^(-BG_MULT)  → con BG_MULT=4 → 6.25% → ~128 bins
# FHSS a -12dB añade ~58 bins reales → total ~186
# WiFi OFDM activa >1800 bins de 2048
# Umbral en 25% = 512 bins separa perfectamente FHSS de WiFi
BG_MULT       = 4
MAX_BINS_FRAC = 0.25
SMOOTH_MS = 0.2

In [ ]:
def _calcular_features(signal, fs, nperseg, bg_mult):
    _, t, Zxx = stft(signal, fs=fs, nperseg=nperseg, return_onesided=False)
    Pxx = np.abs(Zxx) ** 2                          # (N_freq, N_frames)
    # Whitening: cada bin relativo a su mediana temporal
    bg = np.median(Pxx, axis=1, keepdims=True)
    bg[bg < 1e-12] = 1e-12
    Pxx_w = Pxx / bg
    # Feature 1: entropía de Shannon sobre espectro blanqueado
    suma = Pxx_w.sum(axis=0); suma[suma == 0] = 1e-12
    prob = Pxx_w / suma + 1e-12
    H = -(prob * np.log2(prob)).sum(axis=0)
    # Feature 2: bins activos por frame
    # Ruido puro: E[n_active] = N_bins × 2^(-bg_mult)
    # WiFi OFDM: n_active ≈ N_bins
    n_active = (Pxx_w > bg_mult).sum(axis=0).astype(float)
    return t * 1000, H, n_active

def _umbral_robusto(H, z_thresh):
    nf  = np.percentile(H, 90)
    mad = np.median(np.abs(H - np.median(H)))
    ns  = 1.4826 * mad
    return nf - z_thresh * ns, nf, ns

def detectar_bursts_v2(iq_tensor, fs=14e6, nperseg=2048,
                        z_thresh=3.0, min_burst_ms=0.1, merge_gap_ms=1.0,
                        min_z_abs=5.0, bg_mult=4.0, max_bins_frac=0.25,
                        smooth_ms=0.3):
    from scipy.ndimage import gaussian_filter1d

    signal   = iq_tensor[0].numpy() + 1j * iq_tensor[1].numpy()
    t_ms, H, n_active = _calcular_features(signal, fs, nperseg, bg_mult)

    dt = float(t_ms[1] - t_ms[0])

    # ── Suavizado de H ────────────────────────────────────────────────────────
    # Elimina picos de 1-2 frames (WiFi/BT pings) sin destruir bursts sostenidos.
    # smooth_ms=0 desactiva. Umbral y máscara se calculan sobre H_smooth;
    # drop_b y z_peak se calculan sobre H_smooth también para consistencia.
    if smooth_ms > 0:
        sigma_frames = max(0.5, smooth_ms / dt)
        H_smooth = gaussian_filter1d(H, sigma=sigma_frames)
    else:
        H_smooth = H

    umbral, noise_floor, noise_sigma = _umbral_robusto(H_smooth, z_thresh)
    max_bins = nperseg * max_bins_frac

    # ── Máscara + post-procesado temporal ─────────────────────────────────────
    min_f   = max(1, round(min_burst_ms / dt))
    merge_f = max(1, round(merge_gap_ms  / dt))
    mask    = binary_dilation(
                  binary_erosion(H_smooth < umbral, structure=np.ones(min_f)),
                  structure=np.ones(merge_f))
    labeled_arr, n_regions = label(mask)

    bursts = []
    for i in range(1, n_regions + 1):
        idx    = np.where(labeled_arr == i)[0]
        inside = idx[H_smooth[idx] < umbral]
        if len(inside) == 0:
            continue
        i0, i1 = inside[0], inside[-1]

        h_min  = H_smooth[i0:i1+1].min()
        z_peak = (h_min - noise_floor) / (noise_sigma + 1e-10)
        n_act  = float(np.median(n_active[i0:i1+1]))

        # Filtro 1 — significancia estadística
        if abs(z_peak) < min_z_abs:
            continue
        # Filtro 2 — ancho espectral: FHSS = pocos bins, WiFi = casi todos
        if n_act > max_bins:
            continue

        bursts.append({"t0":     t_ms[i0],
                       "t1":     t_ms[i1],
                       "i0":     i0,
                       "i1":     i1,
                       "dur_ms": t_ms[i1] - t_ms[i0],
                       "drop_b": noise_floor - h_min,
                       "z_peak": z_peak,
                       "n_act":  n_act})

    # Devuelve también H_smooth para poder plotear ambas curvas
    return t_ms, H, H_smooth, n_active, bursts, umbral, noise_floor, noise_sigma

# ══════════════════════════════════════════════════════════════════════════════
#  EJECUCIÓN
# ══════════════════════════════════════════════════════════════════════════════
iq, _, _, _ = obtener_una_muestra(DATA_DIR, target=TARGET, snr=SNR, index=INDEX)
t_ms, H, H_smooth, n_active, bursts, umbral, nf, ns = detectar_bursts_v2(
    iq, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    min_z_abs=MIN_Z_ABS, bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    smooth_ms=SMOOTH_MS)
dt  = float(t_ms[1] - t_ms[0])
pfa = norm.sf(Z_THRESH)
max_bins_abs = int(NPERSEG * MAX_BINS_FRAC)
n_ruido_esperado = int(NPERSEG * 2**(-BG_MULT))
print("=" * 65)
print(f"  target={TARGET}  SNR={SNR}dB  index={INDEX}")
print("=" * 65)
print(f"  Δt={dt:.3f}ms/frame  Δf={FS/NPERSEG/1e3:.2f}kHz/bin  N_frames={len(H)}")
print(f"  Noise floor  : {nf:.4f} bits  (P90 whitened)")
print(f"  Noise sigma  : {ns:.4f} bits  (MAD×1.4826)")
print(f"  Umbral entr. : {umbral:.4f} bits  (z={Z_THRESH} → Pfa={pfa*100:.4f}%/frame)")
print(f"  Bins activos → ruido esperado: ~{n_ruido_esperado}  |  límite WiFi: {max_bins_abs}")
print(f"  (FHSS -12dB ≈ {n_ruido_esperado+58} bins | WiFi OFDM ≈ {NPERSEG} bins)")
print()
if not bursts:
    print("  ✗ Sin transmisión detectada")
else:
    print(f"  ✓ {len(bursts)} burst{'s' if len(bursts)!=1 else ''} detectados:")
    for k, b in enumerate(bursts):
        print(f"    B{k+1:02d}: {b['t0']:6.2f}–{b['t1']:6.2f} ms  "
              f"dur={b['dur_ms']:.2f}ms  ↓{b['drop_b']:.3f}b  "
              f"z={b['z_peak']:.1f}  bins_act={b['n_act']:.0f}")
print("=" * 65)
# ══════════════════════════════════════════════════════════════════════════════
#  VISUALIZACIÓN — 4 paneles
# ══════════════════════════════════════════════════════════════════════════════
sig_np = iq[0].numpy() + 1j * iq[1].numpy()
amp    = np.abs(sig_np)
tamp   = np.arange(len(amp)) / FS * 1000
f_st, t_st, Zxx_sp = stft(sig_np, fs=FS, nperseg=NPERSEG, return_onesided=False)
f_sh = np.fft.fftshift(f_st)
Pdb  = 10 * np.log10(np.abs(np.fft.fftshift(Zxx_sp, axes=0))**2 + 1e-12)
COLS = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6','#1abc9c','#ff69b4']
def _marcar(ax, alpha=0.2, yref=None):
    for k, b in enumerate(bursts):
        ax.axvspan(b["t0"], b["t1"], color=COLS[k % len(COLS)], alpha=alpha)
        if yref is not None:
            ax.text((b["t0"]+b["t1"])/2, yref, f'B{k+1}',
                    color=COLS[k % len(COLS)], ha='center', fontsize=8,
                    fontweight='bold', clip_on=True)
plt.style.use('dark_background')
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True,
                          gridspec_kw={'height_ratios': [1, 2, 1.8, 1.2]})
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e'); ax.tick_params(colors='#e0e0e0')
    for sp in ax.spines.values(): sp.set_color('#444')
# P1 — Amplitud
axes[0].plot(tamp[::8], amp[::8], color='#9b59b6', lw=0.4)
_marcar(axes[0], alpha=0.3, yref=amp.max()*0.85)
axes[0].set_title('Amplitud  |I+jQ|', color='white', fontsize=11)
axes[0].set_ylabel('|IQ|', color='white'); axes[0].grid(color='#2c3e6b', alpha=0.3)
# P2 — Espectrograma
axes[1].pcolormesh(t_st*1000, f_sh/1e6, Pdb, shading='gouraud', cmap='viridis')
_marcar(axes[1], alpha=0.25)
axes[1].set_title(f'Espectrograma  Δf={FS/NPERSEG/1e3:.2f}kHz  Δt={dt:.2f}ms',
                  color='white', fontsize=11)
axes[1].set_ylabel('Freq (MHz)', color='white')
# P3 — Entropía + umbral
axes[2].fill_between(t_ms, umbral, nf, alpha=0.08, color='cyan')
axes[2].axhline(nf,     color='white',   ls=':', lw=0.8, alpha=0.5,
                label=f'P90 = {nf:.3f} bits')
axes[2].axhline(umbral, color='#e74c3c', ls='--', lw=1.5,
                label=f'Umbral z={Z_THRESH} → {umbral:.3f} bits  (Pfa={pfa*100:.4f}%)')
axes[2].plot(t_ms, H,        color='#2ecc71', lw=0.6, alpha=0.35, label='H(m) raw')
axes[2].plot(t_ms, H_smooth, color='#2ecc71', lw=1.8,             label=f'H(m) smooth σ={SMOOTH_MS}ms')
for k, b in enumerate(bursts):
    seg = slice(b["i0"], b["i1"]+1)
    axes[2].fill_between(t_ms[seg], H[seg], umbral, where=H[seg] < umbral,
                          color=COLS[k % len(COLS)], alpha=0.55,
                          label=f"B{k+1} {b['dur_ms']:.2f}ms z={b['z_peak']:.1f}")
axes[2].set_title('H(m) whitened + Umbral (P90+MAD)', color='white', fontsize=11)
axes[2].set_ylabel('Bits', color='white')
axes[2].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=7,
               ncol=min(4, 2+len(bursts)), loc='lower left')
axes[2].grid(color='#2c3e6b', alpha=0.3)
# P4 — Bins activos (clave: separa FHSS de WiFi)
axes[3].fill_between(t_ms, 0, n_active, color='#f39c12', alpha=0.7, lw=0,
                      label='Bins activos (Pxx_w > BG_MULT×fondo)')
axes[3].axhline(max_bins_abs, color='#e74c3c', ls='--', lw=1.5,
                label=f'Límite WiFi = {max_bins_abs} bins ({MAX_BINS_FRAC*100:.0f}%)')
axes[3].axhline(n_ruido_esperado, color='white', ls=':', lw=1, alpha=0.5,
                label=f'Ruido esperado = {n_ruido_esperado} bins (2^-{BG_MULT:.0f}×{NPERSEG})')
_marcar(axes[3], alpha=0.3)
axes[3].set_title(
    f'Bins activos por frame  →  FHSS≈{n_ruido_esperado+58} | '
    f'WiFi≈{NPERSEG}  |  Límite={max_bins_abs}',
    color='white', fontsize=11)
axes[3].set_xlabel('Tiempo (ms)', color='white', fontsize=11)
axes[3].set_ylabel('N bins', color='white')
axes[3].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8, loc='upper right')
axes[3].grid(color='#2c3e6b', alpha=0.3)
estado = f"✓ {len(bursts)} bursts" if bursts else "✗ Sin detección"
fig.suptitle(
    f'target={TARGET}  SNR={SNR}dB  index={INDEX}  |  {estado}  |  '
    f'nperseg={NPERSEG}  z={Z_THRESH}  minZ={MIN_Z_ABS}  '
    f'bgMult={BG_MULT}  maxBins={MAX_BINS_FRAC*100:.0f}%',
    color='white', fontsize=11, y=1.005)
plt.tight_layout()
plt.show()

## Prueba con umbral de entropía adaptativo por ventana

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PARÁMETROS
# ══════════════════════════════════════════════════════════════════════════════
TARGET  = 5;  SNR = -14;  INDEX = 20
FS           = 14e6
NPERSEG      = 2048
Z_THRESH     = 3      # más sensible — el filtro de ancho espectral compensa los FA
MIN_BURST_MS = 0.5
MERGE_GAP_MS = 1.0
MIN_Z_ABS    = 4.0
# Discriminador FHSS vs. WiFi:
# P(bin_ruido > BG_MULT × fondo) = 2^(-BG_MULT)  → con BG_MULT=4 → 6.25% → ~128 bins
# FHSS a -12dB añade ~58 bins reales → total ~186
# WiFi OFDM activa >1800 bins de 2048
# Umbral en 25% = 512 bins separa perfectamente FHSS de WiFi
BG_MULT       = 4
MAX_BINS_FRAC = 0.25
SMOOTH_MS = 0.2
ADAPTIVE_WINDOW_MS = 15.0   # ventana de referencia (ms). 0 = umbral global 

In [ ]:
def _calcular_features(signal, fs, nperseg, bg_mult):
    _, t, Zxx = stft(signal, fs=fs, nperseg=nperseg, return_onesided=False)
    Pxx = np.abs(Zxx) ** 2                          # (N_freq, N_frames)
    # Whitening: cada bin relativo a su mediana temporal
    bg = np.median(Pxx, axis=1, keepdims=True)
    bg[bg < 1e-12] = 1e-12
    Pxx_w = Pxx / bg
    # Feature 1: entropía de Shannon sobre espectro blanqueado
    suma = Pxx_w.sum(axis=0); suma[suma == 0] = 1e-12
    prob = Pxx_w / suma + 1e-12
    H = -(prob * np.log2(prob)).sum(axis=0)
    # Feature 2: bins activos por frame
    # Ruido puro: E[n_active] = N_bins × 2^(-bg_mult)
    # WiFi OFDM: n_active ≈ N_bins
    n_active = (Pxx_w > bg_mult).sum(axis=0).astype(float)
    return t * 1000, H, n_active

def _umbral_robusto(H, z_thresh):
    nf  = np.percentile(H, 90)
    mad = np.median(np.abs(H - np.median(H)))
    ns  = 1.4826 * mad
    return nf - z_thresh * ns, nf, ns

def _umbral_adaptativo(H, z_thresh, window_ms, dt):
    """
    Umbral CFAR (Constant False Alarm Rate):
    - P90 calculado en una ventana deslizante de 'window_ms' ms
    - Sigma global (MAD) — más estable que sigma local
    
    Ventajas frente al umbral global:
    - Se adapta a cambios lentos del piso de ruido (AGC, interferencia sostenida)
    - En regiones con WiFi: piso local cae con el WiFi → umbral también cae →
      el dron debe superar una caída EXTRA sobre el WiFi local → más discriminativo
    """
    from scipy.ndimage import percentile_filter
    W = max(3, round(window_ms / dt))   # ventana en frames
    # Piso de ruido local: P90 deslizante
    noise_floor_local = percentile_filter(H, percentile=90, size=W, mode='nearest')
    # Sigma global: MAD sobre todo el registro (más estable que el MAD local)
    mad_global   = np.median(np.abs(H - np.median(H)))
    noise_sigma  = 1.4826 * mad_global
    umbral_local = noise_floor_local - z_thresh * noise_sigma
    return umbral_local, noise_floor_local, noise_sigma

def detectar_bursts_v2(iq_tensor, fs=14e6, nperseg=2048,
                        z_thresh=3.0, min_burst_ms=0.1, merge_gap_ms=1.0,
                        min_z_abs=5.0, bg_mult=4.0, max_bins_frac=0.25,
                        smooth_ms=0.3, adaptive_window_ms=15.0):
    from scipy.ndimage import gaussian_filter1d

    signal = iq_tensor[0].numpy() + 1j * iq_tensor[1].numpy()
    t_ms, H, n_active = _calcular_features(signal, fs, nperseg, bg_mult)
    dt = float(t_ms[1] - t_ms[0])

    # Suavizado opcional
    H_smooth = gaussian_filter1d(H, sigma=max(0.5, smooth_ms / dt)) if smooth_ms > 0 else H

    # Umbral: adaptativo (CFAR) o global
    if adaptive_window_ms > 0:
        umbral, noise_floor, noise_sigma = _umbral_adaptativo(
            H_smooth, z_thresh, adaptive_window_ms, dt)
        umbral_es_vector = True
    else:
        umbral, noise_floor, noise_sigma = _umbral_robusto(H_smooth, z_thresh)
        umbral_es_vector = False

    max_bins = nperseg * max_bins_frac
    min_f    = max(1, round(min_burst_ms / dt))
    merge_f  = max(1, round(merge_gap_ms  / dt))

    mask = binary_dilation(
               binary_erosion(H_smooth < umbral, structure=np.ones(min_f)),
               structure=np.ones(merge_f))
    labeled_arr, n_regions = label(mask)

    # Para z_peak usamos el umbral local en cada punto del burst
    umbral_v = umbral if umbral_es_vector else np.full_like(H_smooth, umbral)
    nf_v     = noise_floor if umbral_es_vector else np.full_like(H_smooth, noise_floor)

    bursts = []
    for i in range(1, n_regions + 1):
        idx    = np.where(labeled_arr == i)[0]
        inside = idx[H_smooth[idx] < umbral_v[idx]]
        if len(inside) == 0:
            continue
        i0, i1 = inside[0], inside[-1]

        # z_peak: caída respecto al umbral LOCAL en el peor punto del burst
        h_seg   = H_smooth[i0:i1+1]
        nf_seg  = nf_v[i0:i1+1]
        umb_seg = umbral_v[i0:i1+1]
        peak_idx   = np.argmin(h_seg)
        h_min      = h_seg[peak_idx]
        local_nf   = nf_seg[peak_idx]
        local_umb  = umb_seg[peak_idx]
        z_peak     = (h_min - local_nf) / (noise_sigma + 1e-10)
        n_act      = float(np.median(n_active[i0:i1+1]))

        if abs(z_peak) < min_z_abs:
            continue
        if n_act > max_bins:
            continue

        bursts.append({"t0": t_ms[i0], "t1": t_ms[i1], "i0": i0, "i1": i1,
                       "dur_ms": t_ms[i1] - t_ms[i0],
                       "drop_b": local_nf - h_min,
                       "z_peak": z_peak, "n_act": n_act})

    return t_ms, H, H_smooth, umbral_v, noise_floor, noise_sigma, n_active, bursts



# ══════════════════════════════════════════════════════════════════════════════
#  EJECUCIÓN
# ══════════════════════════════════════════════════════════════════════════════
iq, _, _, _ = obtener_una_muestra(DATA_DIR, target=TARGET, snr=SNR, index=INDEX)
# ── LLAMADA CORRECTA (añade adaptive_window_ms y orden correcto) ─────────────
t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts_v2(
    iq, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    min_z_abs=MIN_Z_ABS, bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    smooth_ms=SMOOTH_MS, adaptive_window_ms=ADAPTIVE_WINDOW_MS)
# Alias escalares para el diagnóstico (mediana del vector adaptativo)
nf_scalar    = float(np.median(nf_v))
umbral_scalar = float(np.median(umbral_v))
dt  = float(t_ms[1] - t_ms[0])
pfa = norm.sf(Z_THRESH)
max_bins_abs = int(NPERSEG * MAX_BINS_FRAC)
n_ruido_esperado = int(NPERSEG * 2**(-BG_MULT))
print("=" * 65)
print(f"  target={TARGET}  SNR={SNR}dB  index={INDEX}")
print("=" * 65)
print(f"  Δt={dt:.3f}ms/frame  Δf={FS/NPERSEG/1e3:.2f}kHz/bin  N_frames={len(H)}")
print(f"  Noise floor  : {nf_scalar:.4f} bits  (P90 whitened, mediana CFAR)")
print(f"  Noise sigma  : {ns:.4f} bits  (MAD×1.4826)")
print(f"  Umbral entr. : {umbral_scalar:.4f} bits  (z={Z_THRESH} → mediana CFAR)")

print(f"  Bins activos → ruido esperado: ~{n_ruido_esperado}  |  límite WiFi: {max_bins_abs}")
print(f"  (FHSS -12dB ≈ {n_ruido_esperado+58} bins | WiFi OFDM ≈ {NPERSEG} bins)")
print()
if not bursts:
    print("  ✗ Sin transmisión detectada")
else:
    print(f"  ✓ {len(bursts)} burst{'s' if len(bursts)!=1 else ''} detectados:")
    for k, b in enumerate(bursts):
        print(f"    B{k+1:02d}: {b['t0']:6.2f}–{b['t1']:6.2f} ms  "
              f"dur={b['dur_ms']:.2f}ms  ↓{b['drop_b']:.3f}b  "
              f"z={b['z_peak']:.1f}  bins_act={b['n_act']:.0f}")
print("=" * 65)
# ══════════════════════════════════════════════════════════════════════════════
#  VISUALIZACIÓN — 4 paneles
# ══════════════════════════════════════════════════════════════════════════════
sig_np = iq[0].numpy() + 1j * iq[1].numpy()
amp    = np.abs(sig_np)
tamp   = np.arange(len(amp)) / FS * 1000
f_st, t_st, Zxx_sp = stft(sig_np, fs=FS, nperseg=NPERSEG, return_onesided=False)
f_sh = np.fft.fftshift(f_st)
Pdb  = 10 * np.log10(np.abs(np.fft.fftshift(Zxx_sp, axes=0))**2 + 1e-12)
COLS = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6','#1abc9c','#ff69b4']
def _marcar(ax, alpha=0.2, yref=None):
    for k, b in enumerate(bursts):
        ax.axvspan(b["t0"], b["t1"], color=COLS[k % len(COLS)], alpha=alpha)
        if yref is not None:
            ax.text((b["t0"]+b["t1"])/2, yref, f'B{k+1}',
                    color=COLS[k % len(COLS)], ha='center', fontsize=8,
                    fontweight='bold', clip_on=True)
plt.style.use('dark_background')
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True,
                          gridspec_kw={'height_ratios': [1, 2, 1.8, 1.2]})
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e'); ax.tick_params(colors='#e0e0e0')
    for sp in ax.spines.values(): sp.set_color('#444')
# P1 — Amplitud
axes[0].plot(tamp[::8], amp[::8], color='#9b59b6', lw=0.4)
_marcar(axes[0], alpha=0.3, yref=amp.max()*0.85)
axes[0].set_title('Amplitud  |I+jQ|', color='white', fontsize=11)
axes[0].set_ylabel('|IQ|', color='white'); axes[0].grid(color='#2c3e6b', alpha=0.3)
# P2 — Espectrograma
axes[1].pcolormesh(t_st*1000, f_sh/1e6, Pdb, shading='gouraud', cmap='viridis')
_marcar(axes[1], alpha=0.25)
axes[1].set_title(f'Espectrograma  Δf={FS/NPERSEG/1e3:.2f}kHz  Δt={dt:.2f}ms',
                  color='white', fontsize=11)
axes[1].set_ylabel('Freq (MHz)', color='white')
# P3 — Entropía + umbral
# ── PANEL 3 (fill_between usa nf_v vector directamente) ──────────────────────
axes[2].fill_between(t_ms, umbral_v, nf_v, alpha=0.08, color='cyan',
                     label='Zona ruido CFAR')
axes[2].plot(t_ms, nf_v,     color='white',   lw=0.8, ls=':', alpha=0.5,
             label='P90 local (CFAR)')
axes[2].plot(t_ms, umbral_v, color='#e74c3c', lw=1.5, ls='--',
             label=f'Umbral CFAR  z={Z_THRESH}  W={ADAPTIVE_WINDOW_MS}ms')
axes[2].plot(t_ms, H,        color='#2ecc71', lw=0.6, alpha=0.35, label='H(m) raw')
axes[2].plot(t_ms, H_smooth, color='#2ecc71', lw=1.8,             label='H(m) smooth')
for k, b in enumerate(bursts):
    seg = slice(b["i0"], b["i1"]+1)
    axes[2].fill_between(t_ms[seg], H_smooth[seg], umbral_v[seg],
                         where=H_smooth[seg] < umbral_v[seg],
                         color=COLS[k % len(COLS)], alpha=0.55,
                         label=f"B{k+1} {b['dur_ms']:.2f}ms z={b['z_peak']:.1f}")

# P4 — Bins activos (clave: separa FHSS de WiFi)
axes[3].fill_between(t_ms, 0, n_active, color='#f39c12', alpha=0.7, lw=0,
                      label='Bins activos (Pxx_w > BG_MULT×fondo)')
axes[3].axhline(max_bins_abs, color='#e74c3c', ls='--', lw=1.5,
                label=f'Límite WiFi = {max_bins_abs} bins ({MAX_BINS_FRAC*100:.0f}%)')
axes[3].axhline(n_ruido_esperado, color='white', ls=':', lw=1, alpha=0.5,
                label=f'Ruido esperado = {n_ruido_esperado} bins (2^-{BG_MULT:.0f}×{NPERSEG})')
_marcar(axes[3], alpha=0.3)
axes[3].set_title(
    f'Bins activos por frame  →  FHSS≈{n_ruido_esperado+58} | '
    f'WiFi≈{NPERSEG}  |  Límite={max_bins_abs}',
    color='white', fontsize=11)
axes[3].set_xlabel('Tiempo (ms)', color='white', fontsize=11)
axes[3].set_ylabel('N bins', color='white')
axes[3].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8, loc='upper right')
axes[3].grid(color='#2c3e6b', alpha=0.3)
estado = f"✓ {len(bursts)} bursts" if bursts else "✗ Sin detección"
fig.suptitle(
    f'target={TARGET}  SNR={SNR}dB  index={INDEX}  |  {estado}  |  '
    f'nperseg={NPERSEG}  z={Z_THRESH}  minZ={MIN_Z_ABS}  '
    f'bgMult={BG_MULT}  maxBins={MAX_BINS_FRAC*100:.0f}%',
    color='white', fontsize=11, y=1.005)
plt.tight_layout()
plt.show()

## Prueba con RFUAV

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  DETECTOR DE BURSTS — Score unificado (Potencia × Concentración Espectral)
# ══════════════════════════════════════════════════════════════════════════════
# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.signal import stft
# from scipy.ndimage import binary_erosion, binary_dilation, label
# from skimage.filters import threshold_otsu

MODO = "NoisyUAV"   # "NoisyUAV"  o  "RFUAV"

# Config NoisyUAV
TARGET = 1; SNR = -14; INDEX = 21

# Config RFUAV
RUTA_IQ  = r"C:\TFM_data\RFUAV\data\raw\DJI MINI3\DJI MINI3\VTSBW=10\pack2_0-1s.iq"
FS_RFUAV = 10e6; N_MAX = 2_000_000; OFFSET_S = 0.0

# Config detector (físicamente motivados, no dataset-específicos)
NPERSEG      = 4096
MIN_BURST_MS = 0.25
MERGE_GAP_MS = 3.0

# ── CARGA ─────────────────────────────────────────────────────────────────────
if MODO == "NoisyUAV":
    from funciones.cargador import obtener_una_muestra, DATA_DIR
    iq, _, _, _ = obtener_una_muestra(DATA_DIR, target=TARGET, snr=SNR, index=INDEX)
    signal = iq[0].numpy() + 1j * iq[1].numpy()
    FS     = 14e6
    titulo = f"NoisyUAV  target={TARGET}  SNR={SNR}dB"
else:
    import os
    bps  = np.dtype(np.float32).itemsize
    n_sc = min(N_MAX*2, os.path.getsize(RUTA_IQ)//bps - int(OFFSET_S*FS_RFUAV)*2)
    raw  = np.fromfile(RUTA_IQ, dtype=np.float32, count=n_sc,
                       offset=int(OFFSET_S*FS_RFUAV)*2*bps)
    signal = raw[0::2].astype(np.float32) + 1j * raw[1::2].astype(np.float32)
    FS     = FS_RFUAV
    titulo = f"RFUAV  DJI MINI3  BW=10  offset={OFFSET_S}s"
    print(f"Cargado: {len(signal):,} muestras  ({len(signal)/FS*1000:.1f} ms)")

# ── STFT ──────────────────────────────────────────────────────────────────────
_, t_h, Zxx = stft(signal, fs=FS, nperseg=NPERSEG, return_onesided=False)
Pxx         = np.abs(Zxx) ** 2
suma        = Pxx.sum(axis=0); suma[suma == 0] = 1e-12
prob        = Pxx / suma + 1e-12

# Entropía espectral H (bits)
H = -(prob * np.log2(prob)).sum(axis=0)

# Potencia total por frame
P = Pxx.sum(axis=0)

t_ms = t_h * 1000
dt   = float(t_ms[1] - t_ms[0])

# ── SCORE UNIFICADO ───────────────────────────────────────────────────────────
# Normalizamos ambas señales a [0, 1] usando los extremos del propio archivo.
# (1 - H_norm): vale 1 cuando H es mínima (máxima concentración espectral)
# P_norm:       vale 1 cuando hay máxima potencia
#
# Un burst real tiene AMBAS cosas: alta potencia Y espectro concentrado.
# S es el producto → cerca de 1 solo si los DOS features apuntan a lo mismo.
#
# ¿Por qué es universal?
#   RFUAV silencio: P≈0  → P_norm≈0  → S≈0  (aunque H sea baja por ADC noise)
#   NoisyUAV ruido: H≈max → (1-H_norm)≈0 → S≈0  (aunque P sea alta por AWGN)
#   Cualquier burst real: H baja + P alta → S→1

H_norm = (H - H.min()) / (H.max() - H.min() + 1e-12)
# DESPUÉS (dB — da peso justo a todos los bursts):
P_db   = 10 * np.log10(P + 1e-30)
P_norm = (P_db - P_db.min()) / (P_db.max() - P_db.min() + 1e-12)
S      = (1.0 - H_norm) * P_norm   # score en [0, 1]

# Otsu sobre el score
umbral_S = threshold_otsu(S)
mascara_raw = S > umbral_S

# ── SEGMENTACIÓN ──────────────────────────────────────────────────────────────
min_f   = max(1, round(MIN_BURST_MS / dt))
merge_f = max(1, round(MERGE_GAP_MS  / dt))
m_filt  = binary_erosion(mascara_raw, structure=np.ones(min_f))
merged  = binary_dilation(m_filt,     structure=np.ones(merge_f))
labeled, n_bursts = label(merged)

# Paso 4: recortar cada región al núcleo real (solo frames que SÍ superaron el score)
bursts = []
for i in range(1, n_bursts + 1):
    idx_region = np.where(labeled == i)[0]
    # Solo los frames de esta región que estaban REALMENTE en la máscara original
    idx_activo = idx_region[mascara_raw[idx_region]]
    if len(idx_activo) == 0:
        continue
    i0, i1 = idx_activo[0], idx_activo[-1]
    bursts.append({
        "t0":     t_ms[i0],
        "t1":     t_ms[i1],
        "i0":     i0,
        "i1":     i1,
        "dur_ms": t_ms[i1] - t_ms[i0],
        "S_max":  S[i0:i1+1].max(),
    })

# ── DIAGNÓSTICO ───────────────────────────────────────────────────────────────
print("=" * 62)
print(f"  {titulo}")
print("=" * 62)
print(f"  Δt={dt:.3f}ms  Δf={FS/NPERSEG/1e3:.2f}kHz  n_frames={len(H)}")
print(f"  H   : min={H.min():.3f}  max={H.max():.3f}  std={H.std():.4f}")
print(f"  S   : min={S.min():.3f}  max={S.max():.3f}  Otsu={umbral_S:.4f}")
print(f"  Frames burst : {mascara_raw.mean()*100:.1f}%")
print()
estado = f"✓ {n_bursts} burst{'s' if n_bursts!=1 else ''}" if n_bursts else "✗ Sin detección"
print(f"  {estado}")
for k, b in enumerate(bursts):
    print(f"    B{k+1:02d}: {b['t0']:7.2f}–{b['t1']:7.2f} ms  "
          f"dur={b['dur_ms']:.1f}ms  S_max={b['S_max']:.3f}")
print("=" * 62)

# ── VISUALIZACIÓN ─────────────────────────────────────────────────────────────
amp  = np.abs(signal)
tamp = np.arange(len(amp)) / FS * 1000
f_st, t_st, Zxx_sp = stft(signal, fs=FS, nperseg=NPERSEG, return_onesided=False)
f_sh = np.fft.fftshift(f_st)
Pdb  = 10*np.log10(np.abs(np.fft.fftshift(Zxx_sp, axes=0))**2 + 1e-12)
COLS = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6','#1abc9c','#ff69b4']

def _marcar(ax, alpha=0.2, yref=None):
    for k, b in enumerate(bursts):
        col = COLS[k % len(COLS)]
        ax.axvspan(b["t0"], b["t1"], color=col, alpha=alpha)
        if yref is not None:
            ax.text((b["t0"]+b["t1"])/2, yref, f'B{k+1}',
                    color=col, ha='center', fontsize=7, fontweight='bold', clip_on=True)

# Histograma del score
fig_h, ax_h = plt.subplots(figsize=(10, 2.5))
fig_h.patch.set_facecolor('#1a1a2e'); ax_h.set_facecolor('#16213e')
ax_h.hist(S, bins=100, color='#3498db', alpha=0.8)
ax_h.axvline(umbral_S, color='#e74c3c', lw=2, label=f'Otsu = {umbral_S:.4f}')
ax_h.set_title('Histograma del Score  S = (1-H_norm)×P_norm', color='white', fontsize=11)
ax_h.set_xlabel('Score', color='white'); ax_h.tick_params(colors='white')
ax_h.legend(labelcolor='white', facecolor='#1a1a2e', fontsize=9)
plt.tight_layout(); plt.show()

plt.style.use('dark_background')
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True,
                         gridspec_kw={'height_ratios': [1, 2, 1.5, 1.5]})
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e'); ax.tick_params(colors='#e0e0e0')
    for sp in ax.spines.values(): sp.set_color('#444')

axes[0].plot(tamp[::8], amp[::8], color='#9b59b6', lw=0.4)
_marcar(axes[0], alpha=0.3, yref=amp.max()*0.85)
axes[0].set_title('Amplitud', color='white', fontsize=12)
axes[0].set_ylabel('Magnitud', color='white'); axes[0].grid(color='#2c3e6b', alpha=0.3)

axes[1].pcolormesh(t_st*1000, f_sh/1e6, Pdb, shading='gouraud', cmap='viridis')
_marcar(axes[1], alpha=0.25)
axes[1].set_title(f'Espectrograma  nperseg={NPERSEG}', color='white', fontsize=12)
axes[1].set_ylabel('Frecuencia (MHz)', color='white')

axes[2].plot(t_ms, H,      color='#e74c3c', lw=1.0, alpha=0.7, label='H(m)  entropía')
axes[2].plot(t_ms, P_norm*H.max(), color='#f39c12', lw=1.0, alpha=0.7,
             label='P_norm (reescalado a bits)')
_marcar(axes[2], alpha=0.15)
axes[2].set_title('Features individuales', color='white', fontsize=12)
axes[2].set_ylabel('Bits / u.n.', color='white')
axes[2].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8)
axes[2].grid(color='#2c3e6b', alpha=0.3)

axes[3].fill_between(t_ms, 0, S, color='#2ecc71', alpha=0.5, label='Score S')
axes[3].axhline(umbral_S, color='#e74c3c', ls='--', lw=1.8,
                label=f'Otsu = {umbral_S:.4f}')
for k, b in enumerate(bursts):
    axes[3].fill_between(t_ms[b["i0"]:b["i1"]+1], 0,
                         S[b["i0"]:b["i1"]+1],
                         color=COLS[k % len(COLS)], alpha=0.7,
                         label=f'B{k+1} {b["dur_ms"]:.0f}ms')
axes[3].set_title('Score unificado  S = (1-H_norm)×P_norm  +  Otsu automático',
                  color='white', fontsize=12)
axes[3].set_xlabel('Tiempo (ms)', color='white', fontsize=11)
axes[3].set_ylabel('Score [0-1]', color='white')
axes[3].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8,
               ncol=min(4, 2+len(bursts)), loc='upper left')
axes[3].grid(color='#2c3e6b', alpha=0.3)

fig.suptitle(f'{titulo}  |  {estado}  |  nperseg={NPERSEG}  '
             f'Otsu_S={umbral_S:.4f}  minBurst={MIN_BURST_MS}ms  merge={MERGE_GAP_MS}ms',
             color='white', fontsize=11, y=1.005)
plt.tight_layout()
plt.show()


# Prueba artículo: Drone Detection and Classification Using Physical-Layer Protocol Statistical Fingerprint

In [ ]:
# ── PARÁMETROS ────────────────────────────────────────────────────────────────
TARGET = 1;  SNR = 10;  INDEX = 22

FS     = 14e6
NPERSEG = 4096          # para el espectrograma visual

# PSE (Bartlett)
N_PSE  = 2048           # bins del Bartlett estimator

# LPF sobre envolvente (extrae comportamiento temporal del protocolo)
F_LPF  = 10e3           # Hz — corte del filtro

# Histéresis (sin unidades — normalizados al midpoint)
MU_LOW  = 0.44          # umbral de apagado
MU_HIGH = 0.56          # umbral de encendido  (MU_HIGH > MU_LOW = histéresis)

# Post-procesado
MIN_BURST_MS = 0.3      # duración mínima de burst válido

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  DETECTOR  — Morge-Rollet 2022: PSE global + LPF + Histéresis en amplitud
#  Ref: doi:10.3390/s22176701
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, stft
from scipy.ndimage import label

# ── FIN PARÁMETROS ────────────────────────────────────────────────────────────

import sys; sys.path.insert(0, r"c:\repos\DroneDetectionRF\NoisyUAV")
from funciones.cargador import obtener_una_muestra, DATA_DIR

iq, _, _, _ = obtener_una_muestra(DATA_DIR, target=TARGET, snr=SNR, index=INDEX)
signal = iq[0].numpy() + 1j * iq[1].numpy()

# ══════════════════════════════════════════════════════════════════════════════
#  PASO 1 — PSE global (Bartlett, N=2048)
#  Un único escalar por ventana → ¿hay señal en este fragmento?
# ══════════════════════════════════════════════════════════════════════════════
L = len(signal)
K = L // N_PSE                             # número de sub-ventanas
psd = np.zeros(N_PSE)
for k in range(K):
    seg  = signal[k*N_PSE:(k+1)*N_PSE]
    fft2 = np.abs(np.fft.fft(seg)) ** 2
    psd += fft2 / K                        # promedio → estimador de Bartlett

psd_norm = psd / psd.sum() + 1e-12
PSE = float(-(psd_norm * np.log2(psd_norm)).sum())

# Umbral empírico: Morge-Rollet lo fijan en η=10.9992 (1er percentil del ruido)
# Como no tenemos esa distribución a mano, usamos el valor del paper directamente.
ETA = 10.9992
pse_decision = "✓ SEÑAL detectada" if PSE < ETA else "✗ Solo ruido (PSE > η)"
print(f"PSE global = {PSE:.4f}  |  η = {ETA}  |  {pse_decision}")

# ══════════════════════════════════════════════════════════════════════════════
#  PASO 2 — LPF sobre envolvente  +  Histéresis → segmentación de paquetes
# ══════════════════════════════════════════════════════════════════════════════
# 2a. Envolvente
env = np.abs(signal)

# 2b. LPF a F_LPF Hz
b, a = butter(4, F_LPF / (FS / 2), btype='low')
z    = filtfilt(b, a, env)          # z[n]: comportamiento temporal lento

# 2c. Estadísticos de escala (self-calibrating)
minz  = z.min(); maxz  = z.max()
z_bar = (minz + maxz) / 2.0        # punto medio

thr_on  = minz + MU_HIGH * z_bar   # umbral de activación
thr_off = minz + MU_LOW  * z_bar   # umbral de desactivación (< thr_on)

print(f"Envolvente LPF → min={minz:.4f}  max={maxz:.4f}  midpoint={z_bar:.4f}")
print(f"Umbrales        → ON={thr_on:.4f}  OFF={thr_off:.4f}  Δ={thr_on-thr_off:.4f}")

# 2d. Máquina de estados (Algoritmo 1, Morge-Rollet)
t_mask = np.zeros(len(z), dtype=bool)
state  = False                      # empezamos en "sin señal"
for i in range(1, len(z)):
    if not state and z[i] >  thr_on:   state = True
    elif state  and z[i] <  thr_off:   state = False
    t_mask[i] = state

# 2e. Filtro de duración mínima
min_samp  = max(1, round(MIN_BURST_MS * 1e-3 * FS))
labeled_arr, n_regions = label(t_mask)

bursts = []
for i in range(1, n_regions + 1):
    idx = np.where(labeled_arr == i)[0]
    if len(idx) < min_samp:
        continue
    i0, i1 = idx[0], idx[-1]
    t0 = i0 / FS * 1000
    t1 = i1 / FS * 1000
    bursts.append({"t0": t0, "t1": t1, "i0": i0, "i1": i1,
                   "dur_ms": t1 - t0})

n_bursts = len(bursts)
print(f"\n{'✓' if n_bursts else '✗'} {n_bursts} burst{'s' if n_bursts!=1 else ''} detectados:")
for k, b in enumerate(bursts):
    print(f"  B{k+1:02d}: {b['t0']:7.2f}–{b['t1']:7.2f} ms  dur={b['dur_ms']:.2f} ms")

# ══════════════════════════════════════════════════════════════════════════════
#  VISUALIZACIÓN — 4 paneles
# ══════════════════════════════════════════════════════════════════════════════
tamp = np.arange(L) / FS * 1000     # eje temporal muestras → ms
t_z  = tamp                          # mismo eje para la envolvente

f_st, t_st, Zxx_sp = stft(signal, fs=FS, nperseg=NPERSEG, return_onesided=False)
f_sh = np.fft.fftshift(f_st)
Pdb  = 10 * np.log10(np.abs(np.fft.fftshift(Zxx_sp, axes=0))**2 + 1e-12)

COLS = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6','#1abc9c','#ff69b4']

def _marcar(ax, alpha=0.2, yref=None):
    for k, b in enumerate(bursts):
        col = COLS[k % len(COLS)]
        ax.axvspan(b["t0"], b["t1"], color=col, alpha=alpha)
        if yref is not None:
            ax.text((b["t0"]+b["t1"])/2, yref, f'B{k+1}',
                    color=col, ha='center', fontsize=8, fontweight='bold', clip_on=True)

plt.style.use('dark_background')
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True,
                         gridspec_kw={'height_ratios': [1, 2, 1.5, 1.5]})
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e'); ax.tick_params(colors='#e0e0e0')
    for sp in ax.spines.values(): sp.set_color('#444')

# P1 — Amplitud cruda
axes[0].plot(tamp[::8], np.abs(signal)[::8], color='#9b59b6', lw=0.4)
_marcar(axes[0], alpha=0.3, yref=np.abs(signal).max()*0.85)
axes[0].set_title('Amplitud  |I+jQ|  (cruda)', color='white', fontsize=11)
axes[0].set_ylabel('|IQ|', color='white')
axes[0].grid(color='#2c3e6b', alpha=0.3)

# P2 — Espectrograma
axes[1].pcolormesh(t_st*1000, f_sh/1e6, Pdb, shading='gouraud', cmap='viridis')
_marcar(axes[1], alpha=0.25)
axes[1].set_title(f'Espectrograma  nperseg={NPERSEG}  Δf={FS/NPERSEG/1e3:.2f} kHz',
                  color='white', fontsize=11)
axes[1].set_ylabel('Freq (MHz)', color='white')

# P3 — Envolvente filtrada + umbrales
axes[2].plot(t_z[::4], z[::4], color='#f39c12', lw=0.8, label='Envolvente LPF z[n]')
axes[2].axhline(thr_on,  color='#2ecc71', ls='--', lw=1.5,
                label=f'μ_high = {MU_HIGH}  →  thr_ON  = {thr_on:.4f}')
axes[2].axhline(thr_off, color='#e74c3c', ls='--', lw=1.5,
                label=f'μ_low  = {MU_LOW}   →  thr_OFF = {thr_off:.4f}')
axes[2].axhline(z_bar,   color='white',   ls=':', lw=0.8, alpha=0.5,
                label=f'midpoint z̄ = {z_bar:.4f}')
_marcar(axes[2], alpha=0.2)
axes[2].set_title(f'Envolvente filtrada  (LPF Fpass={F_LPF/1e3:.0f} kHz)  +  Umbrales de histéresis',
                  color='white', fontsize=11)
axes[2].set_ylabel('Amplitud', color='white')
axes[2].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8, loc='upper right')
axes[2].grid(color='#2c3e6b', alpha=0.3)

# P4 — Máscara binaria resultante
axes[3].fill_between(t_z[::4], 0, t_mask[::4].astype(float),
                     color='#3498db', alpha=0.6, label='Máscara histéresis')
for k, b in enumerate(bursts):
    col = COLS[k % len(COLS)]
    axes[3].fill_between(
        tamp[b["i0"]:b["i1"]+1], 0, 1,
        color=col, alpha=0.7,
        label=f"B{k+1} {b['dur_ms']:.1f}ms")
axes[3].set_ylim(-0.1, 1.2)
axes[3].set_title('Máscara de detección  (histéresis)', color='white', fontsize=11)
axes[3].set_xlabel('Tiempo (ms)', color='white', fontsize=11)
axes[3].set_ylabel('Estado', color='white')
axes[3].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8,
               ncol=min(4, 1+n_bursts), loc='upper left')
axes[3].grid(color='#2c3e6b', alpha=0.3)

estado = f"✓ {n_bursts} bursts" if n_bursts else "✗ Sin detección"
fig.suptitle(
    f'target={TARGET}  SNR={SNR}dB  index={INDEX}  |  '
    f'PSE={PSE:.4f} [{pse_decision[:15]}]  |  {estado}  |  '
    f'LPF={F_LPF/1e3:.0f}kHz  μH={MU_HIGH}  μL={MU_LOW}  minBurst={MIN_BURST_MS}ms',
    color='white', fontsize=10, y=1.005)
plt.tight_layout()
plt.show()
